# Bank Marketing Classification

---

Author: heleneInsights

---

## Objective

This project applies data cleaning, exploratory data analysis, and machine learning models to predict whether a client will subscribe to a term deposit based on the **Bank Marketing Dataset** from the UCI Machine Learning Repository.

---

## Problem Statement
The goal is to build a classification model that predicts the variable **`y`** (whether a client subscribes to a term deposit) using demographic, financial, and campaign-related features.

# Dataset Description

This dataset contains information from direct phone marketing campaigns run by a Portuguese bank. Each record represents a client contact, often with multiple calls per client. The goal is to predict whether a client will subscribe to a term deposit (“yes” or “no”) based on their characteristics and campaign interactions.

## Dataset Information

* **Number of observations**: 4,119 client records
* **Number of features**: 20 input variables + 1 target variable
* **Target variable**: `y` (term deposit subscription: yes/no)

## Data Dictionary — Bank Marketing Dataset

| Column         | Type  | Description                           | Example             |
| -------------- | ----- | ------------------------------------- | ------------------- |
| age            | int   | Age of the client                     | 39                  |
| job            | str   | Type of job                           | "admin."            |
| marital        | str   | Marital status                        | "married"           |
| education      | str   | Education level                       | "university.degree" |
| default        | str   | Has credit in default?                | "no"                |
| housing        | str   | Has housing loan?                     | "yes"               |
| loan           | str   | Has personal loan?                    | "no"                |
| contact        | str   | Contact communication type            | "cellular"          |
| month          | str   | Last contact month                    | "may"               |
| day_of_week    | str   | Last contact day                      | "fri"               |
| duration       | int   | Last call duration (seconds)          | 487                 |
| campaign       | int   | Number of contacts during campaign    | 2                   |
| pdays          | int   | Days since last contact (999 = never) | 999                 |
| previous       | int   | Previous campaign contacts            | 0                   |
| poutcome       | str   | Previous campaign outcome             | "nonexistent"       |
| emp.var.rate   | float | Employment variation rate             | -1.8                |
| cons.price.idx | float | Consumer price index                  | 92.893              |
| cons.conf.idx  | float | Consumer confidence index             | -46.2               |
| euribor3m      | float | Euribor 3-month rate                  | 1.313               |
| nr.employed    | float | Number of employees indicator         | 5099.1              |
| y              | str   | Target: subscribed term deposit?      | "no"                |

For further information regarding the dataset exploration and the rationale behind the preprocessing choices, please refer to `01_eda.ipynb`.

---

# Data Loading and Inspection
## Import Libraries

In [ ]:
# Standard library imports
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from scipy.stats import gaussian_kde

from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    OneHotEncoder, 
    OrdinalEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    train_test_split
) 

import joblib

### Dataset Availability and Path Setup

In [2]:
data_dir = Path("../data/raw/")

# List available files (for verification)
available_files = list(data_dir.iterdir())
print("Available files:", available_files)
file_name = "bank-additional.csv"
dataset_path = data_dir / file_name

Available files: [WindowsPath('../data/raw/bank-additional.csv')]


In [3]:
# Validate dataset existence
if not dataset_path.exists():
    raise FileNotFoundError(f"{file_name} not found in {data_dir}")

print("Dataset found at:", dataset_path)

Dataset found at: ..\data\raw\bank-additional.csv


## Load Dataset

In [4]:
df = pd.read_csv(dataset_path, sep=";")

### Display Configuration

To prevent pandas from truncating large outputs during data exploration, the following display settings were applied:

In [5]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

This allows all rows and columns to be displayed when inspecting DataFrames.

### Verify Data Load

The dataset was successfully loaded and inspected to confirm that all features were correctly parsed and available for preprocessing. A sample of the first five records is shown below.

*For further information regarding the dataset structure, feature distributions, data quality assessment, and exploratory findings, please refer to `01_eda.ipynb`.*

In [6]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,487,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,346,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,227,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,17,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,58,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


### Verify Structure

The dataset `bank-additional.csv` was successfully loaded and its structure was verified.

In [7]:
print(f"{file_name}")
print(f"Shape: {df.shape}")
print(f"→ {df.shape[0]} rows, {df.shape[1]} columns\n")

bank-additional.csv
Shape: (4119, 21)
→ 4119 rows, 21 columns



This confirms that the dataset contains 4,119 observations and 21 variables available for preprocessing and subsequent modeling.

*For further information regarding the dataset structure, feature descriptions, data quality assessment, and exploratory analysis, please refer to `01_eda.ipynb`.*

### Inspect Data Types

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   str    
 2   marital         4119 non-null   str    
 3   education       4119 non-null   str    
 4   default         4119 non-null   str    
 5   housing         4119 non-null   str    
 6   loan            4119 non-null   str    
 7   contact         4119 non-null   str    
 8   month           4119 non-null   str    
 9   day_of_week     4119 non-null   str    
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   str    
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   float64
 18 

The dataset contains **10 numerical features** (`int64` and `float64`) and **11 categorical features** (`str`). 

*See `01_eda.ipynb` for additional details.*

---

# Planned Feature Preprocessing

Before model training, the dataset will undergo preprocessing to ensure variables are in a suitable format for machine learning algorithms and to minimize the impact of skewness, outliers, and hidden missing values.

### Planned Steps

* Remove non-informative variables when appropriate.
* Convert the target variable (`y`) into a binary label using **LabelEncoder** (`no=0`, `yes=1`).
* Handle hidden missing values represented by `"unknown"` before encoding.
* Apply **RobustScaler** to highly skewed numerical variables with significant outliers.
* Apply **StandardScaler** to numerical variables with approximately symmetric distributions and no major outlier issues.
* Apply **MinMaxScaler** to non-normal numerical variables without significant outliers.
* Encode nominal categorical variables using **One-Hot Encoding (`drop='first'`)**.
* Encode ordinal categorical variables using **OrdinalEncoder**.

| Column           | Variable Type       | Planned Treatment                                      | Encoding / Scaling Strategy       | Reason                                                   |
| ---------------- | ------------------- | ------------------------------------------------------ | --------------------------------- | -------------------------------------------------------- |
| `age`            | Numerical           | Keep                                                   | RobustScaler                  | Contains outliers and mild skewness                      |
| `job`            | Nominal Categorical | Keep                                                   | One-Hot Encoding (`drop='first'`) | No natural order                                         |
| `marital`        | Nominal Categorical | Keep                                                   | One-Hot Encoding (`drop='first'`) | No natural order                                         |
| `education`      | Ordinal Categorical | Keep                                                   | OrdinalEncoder                    | Natural educational progression                          |
| `default`        | Nominal Categorical | Handle `unknown` values                                | One-Hot Encoding (`drop='first'`) | No natural order                                         |
| `housing`        | Nominal Categorical | Handle `unknown` values                                | One-Hot Encoding (`drop='first'`) | Binary categorical variable                              |
| `loan`           | Nominal Categorical | Handle `unknown` values                                | One-Hot Encoding (`drop='first'`) | Binary categorical variable                              |
| `contact`        | Nominal Categorical | Keep                                                   | One-Hot Encoding (`drop='first'`) | No natural order                                         |
| `month`          | Ordinal Categorical | Keep                                                   | OrdinalEncoder                    | Temporal ordering                                        |
| `day_of_week`    | Ordinal Categorical | Keep                                                   | OrdinalEncoder                    | Temporal ordering                                        |
| `duration`       | Numerical           | Keep for benchmark models; remove for realistic models | RobustScaler                  | Highly skewed with significant outliers and leakage risk |
| `campaign`       | Numerical           | Keep                                                   | RobustScaler                  | Strong right skew and outliers                           |
| `pdays`          | Numerical           | Transform special value (999) before modeling          | RobustScaler                  | Highly skewed and contains outliers                      |
| `previous`       | Numerical           | Keep                                                   | RobustScaler                  | Highest proportion of outliers                           |
| `poutcome`       | Nominal Categorical | Keep                                                   | One-Hot Encoding (`drop='first'`) | No natural order                                         |
| `emp.var.rate`   | Numerical           | Keep                                                   | MinMaxScaler                      | No outliers detected; non-normal distribution            |
| `cons.price.idx` | Numerical           | Keep                                                   | MinMaxScaler                      | No outliers detected; non-normal distribution            |
| `cons.conf.idx`  | Numerical           | Keep                                                   | RobustScaler                  | Outliers detected and non-normal distribution            |
| `euribor3m`      | Numerical           | Keep                                                   | MinMaxScaler                      | No outliers detected; non-normal distribution            |
| `nr.employed`    | Numerical           | Keep                                                   | MinMaxScaler                      | No outliers detected; non-normal distribution            |
| `y`              | Target Variable     | Convert to binary                                      | LabelEncoder                      | Required for binary classification                       |


### Additional Notes

* The value `"unknown"` appears in several categorical variables and represents hidden missing data. Different strategies (keeping as a category, imputation, or removal) will be evaluated during preprocessing.
* The variable `duration` is known only after a call has ended and may cause **data leakage**. Two modeling approaches may be considered:

  * **Benchmark model:** includes `duration`.
  * **Real-world predictive model:** excludes `duration`.
* The variable `pdays` contains a special value (`999`) that does not represent a true number of days and should be treated separately before scaling.

---

# Data Cleaning
## Missing Values Handling

The dataset contains no explicit missing values (`NaN`). However, several categorical features use the value `"unknown"` to represent unavailable information. These entries will be treated as hidden missing values and handled before feature encoding.

*For a detailed analysis of missing-value patterns and feature distributions, refer to `01_eda.ipynb`.*

In [10]:
unknown_summary = pd.DataFrame({
    'Unknown Values': (df == 'unknown').sum(),
    'Percentage (%)': ((df == 'unknown').sum() / len(df) * 100).round(2)
})

unknown_summary.sort_values('Unknown Values', ascending=False)

,Unknown Values,Percentage (%)
default,803,19.50
education,167,4.05
housing,105,2.55
loan,105,2.55
job,39,0.95
marital,11,0.27
age,0,0.00
contact,0,0.00
month,0,0.00
day_of_week,0,0.00


The dataset contains **no explicit missing values (`NaN`)**. However, several categorical features use the value `"unknown"` to represent unavailable information.

| Feature     | Unknown Values | Percentage (%) |
| ----------- | -------------: | -------------: |
| `default`   |            803 |          19.50 |
| `education` |            167 |           4.05 |
| `housing`   |            105 |           2.55 |
| `loan`      |            105 |           2.55 |
| `job`       |             39 |           0.95 |
| `marital`   |             11 |           0.27 |

The highest proportion of hidden missing values occurs in `default` (19.5%), while the remaining affected features contain relatively few `"unknown"` entries. These values will be treated as hidden missing data and addressed during preprocessing before feature encoding.

### Missing Value Handling Decision Matrix

| Feature     | Unknown % | Treatment        |
| ----------- | --------: | ---------------------------- |
| `default`   |    19.50% | Keep `"unknown"`             |
| `education` |     4.05% | Keep `"unknown"`             |
| `housing`   |     2.55% | Impute with mode (`yes`)     |
| `loan`      |     2.55% | Impute with mode (`no`)      |
| `job`       |     0.95% | Keep `"unknown"`             |
| `marital`   |     0.27% | Impute with mode (`married`) |

The reasoning is straightforward:

* **`job` and `education`** have many categories. Replacing missing values with the mode would artificially inflate `admin.` and `university.degree`, respectively.
* **`default`** has nearly 20% unknown values. Imputation would overwrite a substantial portion of the dataset and likely distort the feature.
* **`housing`, `loan`, and `marital`** have very low missingness and only a few categories, making mode imputation reasonable and low risk.

### Handling Missing Values in Categorical Features

Hidden missing values represented by `"unknown"` will be handled according to the characteristics of each feature.

* `housing`, `loan`, and `marital` will be imputed using the **most frequent category (mode)** due to their low proportion of missing values.
* For `job`, `education`, and `default`, the `"unknown"` category will be retained, as these features contain multiple categories and the missingness itself may carry useful information.

This approach preserves all observations while avoiding unnecessary assumptions about missing categorical values.

In [12]:
# Create a copy of the dataset
df_categorical_imputed = df.copy()

# Features to impute using the mode
categorical_mode_cols = [
    "housing",
    "loan",
    "marital"
]

for col in categorical_mode_cols:

    values_without_unknown = (
        df_categorical_imputed[col]
        .loc[df_categorical_imputed[col] != "unknown"]
    )

    most_frequent_category = (
        values_without_unknown.mode()[0]
    )

    df_categorical_imputed[col] = (
        df_categorical_imputed[col]
        .replace(
            "unknown",
            most_frequent_category
        )
    )
    
# Verify that no 'unknown' values remain
(df_categorical_imputed[categorical_mode_cols] == "unknown").sum()

housing    0
loan       0
marital    0
dtype: int64

### Check Missing Values After Cleaning

In [14]:
unknown_summary = pd.DataFrame({
    'Unknown Values': (df_categorical_imputed == 'unknown').sum(),
    'Percentage (%)': ((df_categorical_imputed == 'unknown').sum() / len(df_categorical_imputed) * 100).round(2)
})

unknown_summary.sort_values('Unknown Values', ascending=False)

,Unknown Values,Percentage (%)
default,803,19.50
education,167,4.05
job,39,0.95
age,0,0.00
marital,0,0.00
housing,0,0.00
loan,0,0.00
contact,0,0.00
month,0,0.00
day_of_week,0,0.00


Mode imputation successfully removed all `"unknown"` values from `housing`, `loan`, and `marital`.

The remaining `"unknown"` categories were intentionally retained in `default`, `education`, and `job`, where missingness may carry useful information and imputation would require stronger assumptions.

| Feature     | Remaining `"unknown"` Values | Percentage (%) |
| ----------- | ---------------------------: | -------------: |
| `default`   |                          803 |          19.50 |
| `education` |                          167 |           4.05 |
| `job`       |                           39 |           0.95 |

No hidden missing values remain in the features selected for imputation.

## Handling Special Values

The feature `pdays` uses the value `999` to indicate that a customer was not previously contacted. Since this value does not represent an actual number of days, it will be treated separately before scaling and model training.

In [15]:
df_categorical_imputed['pdays'].value_counts(dropna=False)

pdays
999    3959
3        52
6        42
4        14
7        10
10        8
12        5
5         4
2         4
1         3
9         3
18        2
15        2
0         2
16        2
13        2
11        1
19        1
17        1
21        1
14        1
Name: count, dtype: int64

In [ ]:
df_pdays = df_categorical_imputed.copy()

# Indicator for whether the client was previously contacted
df_pdays["previously_contacted"] = (
    df_pdays["pdays"] != 999
).astype(int)

# Replace the special value
df_pdays["pdays"] = df_pdays["pdays"].replace(999, 0)

In [17]:
df_pdays['pdays'].value_counts(dropna=False)

pdays
0     3961
3       52
6       42
4       14
7       10
10       8
12       5
5        4
2        4
1        3
9        3
18       2
15       2
16       2
13       2
11       1
19       1
17       1
21       1
14       1
Name: count, dtype: int64

The feature `pdays` originally used the value `999` to indicate that a client had not been previously contacted. Since this value does not represent an actual number of days, it was replaced with `0`, allowing the feature to distinguish between clients who had never been contacted and those contacted within the previous 1–21 days.

This transformation preserves the original meaning of the variable while avoiding the influence of an artificial extreme value during scaling and model training.

## Column Type Validation
Features were checked to ensure type consistency. Column types were standardized before defining the final modeling dataset.

In [48]:
df_pdays.dtypes

age                       int64
job                         str
marital                     str
education                   str
default                     str
housing                     str
loan                        str
contact                     str
month                       str
day_of_week                 str
duration                  int64
campaign                  int64
pdays                     int64
previous                  int64
poutcome                    str
emp.var.rate            float64
cons.price.idx          float64
cons.conf.idx           float64
euribor3m               float64
nr.employed             float64
y                           str
previously_contacted      int64
dtype: object

No type inconsistencies or unexpected object encodings were detected after preprocessing.

## Data Type Validation

In [49]:
for col in df_pdays.columns:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(f"{'='*40}")

    values_types = pd.DataFrame({
        "Value": df_pdays[col].unique()
    })

    values_types["Type"] = values_types["Value"].apply(lambda x: type(x).__name__)

    print(values_types.sort_values("Type"))


Column: age
    Value Type
0      30  int
35     57  int
36     43  int
37     53  int
38     75  int
39     82  int
40     71  int
41     21  int
42     22  int
43     23  int
44     26  int
45     81  int
46     61  int
47     67  int
48     73  int
34     59  int
49     18  int
51     74  int
52     77  int
53     86  int
54     85  int
55     63  int
56     88  int
57     78  int
58     72  int
59     68  int
60     80  int
61     66  int
62     19  int
63     62  int
64     65  int
50     64  int
65     69  int
33     54  int
31     42  int
1      39  int
2      25  int
3      38  int
4      47  int
5      32  int
6      41  int
7      31  int
8      35  int
9      36  int
10     29  int
11     27  int
12     44  int
13     46  int
14     45  int
32     49  int
15     50  int
17     40  int
18     28  int
19     34  int
20     33  int
21     51  int
22     48  int
23     20  int
24     76  int
25     56  int
26     24  int
27     58  int
28     60  int
29     37  int
30     52  i

All variables were checked for type consistency and value validity after preprocessing. Categorical variables contain expected discrete levels, while numerical variables remain correctly formatted as integer or float types. No unexpected type coercion or mixed-type columns were detected.

The final dataset includes:

* Numerical features (int64, float64)
* Encoded binary feature (`previously_contacted`)
* Clean categorical variables ready for encoding

## Distributions After Cleaning
### Numeric
#### Histograms

In [35]:
features = {
    "pdays (Original DF)": df["pdays"],
    "pdays (Filtered DF)": df_pdays["pdays"],
    "previously_contacted": df_pdays["previously_contacted"]
}

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=list(features.keys())
)

for i, (name, series) in enumerate(features.items(), start=1):

    data = series.dropna()

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=data,
            nbinsx=30,
            opacity=0.6,
            showlegend=False
        ),
        row=1,
        col=i
    )

    # KDE only if enough unique values
    if data.nunique() > 1:

        kde = gaussian_kde(data)

        x_range = np.linspace(
            data.min(),
            data.max(),
            200
        )

        kde_values = kde(x_range)

        fig.add_trace(
            go.Scatter(
                x=x_range,
                y=kde_values * len(data) * (data.max() - data.min()) / 30,
                mode="lines",
                line=dict(width=2),
                showlegend=False
            ),
            row=1,
            col=i
        )

    # Mean
    fig.add_vline(
        x=data.mean(),
        line_dash="dash",
        line_color="green",
        annotation_text="Mean",
        annotation_position="right",
        row=1,
        col=i
    )

    # Median
    fig.add_vline(
        x=data.median(),
        line_dash="dot",
        line_color="red",
        annotation_text="Median",
        annotation_position="bottom right",
        row=1,
        col=i
    )

fig.update_layout(
    title="Comparison of pdays and previously_contacted",
    template="plotly_white",
    width=1500,
    height=500,
    showlegend=False
)

fig.update_xaxes(title_text="Value")
fig.update_yaxes(title_text="Frequency")

fig.show()

##### `pdays`

After replacing the special value `999` with `0`, the distribution of `pdays` becomes highly concentrated at zero, with a long right tail extending to 21 days. This reflects the fact that most clients were not previously contacted, while a small subset had been contacted within the preceding three weeks.

##### `previously_contacted`

The newly created feature `previously_contacted` exhibits a highly imbalanced binary distribution. Most observations belong to the `0` category (not previously contacted), while only a small proportion of clients are assigned to the `1` category (previously contacted).

The transformation of `pdays` produces a highly right-skewed distribution with a dominant peak at zero. The derived feature `previously_contacted` is strongly imbalanced, with approximately 96% of observations equal to 0 and 4% equal to 1.

### Categorical
#### Categorical Value Consistency

In [36]:
categorical_cols = [
    "housing",
    "loan",
    "marital"
]

for col in categorical_cols:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(f"{'='*40}")

    summary = (
        df_pdays[col]
        .value_counts(dropna=False)
        .reset_index()
    )

    summary.columns = [col, "Count"]

    summary["Percentage"] = (
        summary["Count"] / len(df_pdays) * 100
    ).round(2)

    print(summary)


Column: housing
  housing  Count  Percentage
0     yes   2280       55.35
1      no   1839       44.65

Column: loan
  loan  Count  Percentage
0   no   3454       83.86
1  yes    665       16.14

Column: marital
    marital  Count  Percentage
0   married   2520       61.18
1    single   1153       27.99
2  divorced    446       10.83


In [ ]:
categorical_cols = ["housing", "loan", "marital"]

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[
        "df - housing", "df - loan", "df - marital",
        "df_pdays - housing", "df_pdays - loan", "df_pdays - marital"
    ],
    horizontal_spacing=0.12,
    vertical_spacing=0.25
)

# ---------- TOP ROW: df ----------
for i, feature in enumerate(categorical_cols, start=1):

    counts = df[feature].astype(str).value_counts()

    fig.add_trace(
        go.Bar(
            x=counts.index,
            y=counts.values,
            name="df",
            showlegend=(i == 1)
        ),
        row=1,
        col=i
    )

    fig.update_xaxes(
        title_text=feature,
        tickangle=45,
        row=1,
        col=i
    )

    fig.update_yaxes(
        title_text="Count",
        row=1,
        col=i
    )

# ---------- BOTTOM ROW: df_pdays ----------
for i, feature in enumerate(categorical_cols, start=1):

    counts = df_pdays[feature].astype(str).value_counts()

    fig.add_trace(
        go.Bar(
            x=counts.index,
            y=counts.values,
            name="df_pdays",
            showlegend=(i == 1)
        ),
        row=2,
        col=i
    )

    fig.update_xaxes(
        title_text=feature,
        tickangle=45,
        row=2,
        col=i
    )

    fig.update_yaxes(
        title_text="Count",
        row=2,
        col=i
    )

# ---------- layout ----------
fig.update_layout(
    title="Comparison: df (top) vs df_pdays (bottom)",
    template="plotly_white",
    height=800,
    width=1100,
    showlegend=True
)

fig.show()

As expected, mode imputation slightly increases the frequency of the dominant category in `housing`, `loan`, and `marital`, but does not materially change the overall class distributions due to the low proportion of missing values.

### Countplots
##### IQR calculation

In [43]:
feature_columns = ["pdays", "previously_contacted"]

outlier_summary = []

for col in feature_columns:

    Q1 = df_pdays[col].quantile(0.25)
    Q3 = df_pdays[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_outliers = (
        (df_pdays[col] < lower_bound) |
        (df_pdays[col] > upper_bound)
    ).sum()

    outlier_summary.append({
        "Feature": col,
        "Outliers": n_outliers,
        "Percentage": round(n_outliers / len(df_pdays) * 100, 2)
    })

outlier_df = (
    pd.DataFrame(outlier_summary)
    .sort_values(by="Percentage", ascending=False)
)

outlier_df

,Feature,Outliers,Percentage
1,previously_contacted,160,3.88
0,pdays,158,3.84


In [44]:
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "df - pdays",
        "df_pdays - pdays",
        "df_pdays - previously_contacted"
    ]
)

# --- df pdays ---
fig.add_trace(
    go.Box(
        y=df["pdays"],
        name="df",
        boxmean=True
    ),
    row=1,
    col=1
)

# --- df_pdays pdays ---
fig.add_trace(
    go.Box(
        y=df_pdays["pdays"],
        name="df_pdays",
        boxmean=True
    ),
    row=1,
    col=2
)

# --- previously_contacted ---
fig.add_trace(
    go.Box(
        y=df_pdays["previously_contacted"],
        name="previously_contacted",
        boxmean=True
    ),
    row=1,
    col=3
)

# Layout
fig.update_layout(
    title="Boxplot Comparison: pdays vs previously_contacted",
    template="plotly_white",
    width=1200,
    height=450,
    showlegend=False
)

fig.update_yaxes(title_text="Values")

fig.show()

After cleaning, the distribution of `pdays` becomes heavily concentrated at 0, representing clients that were not previously contacted. The remaining values (1–21) form a sparse right tail, reflecting clients with prior contact history.

The derived feature `previously_contacted` is a binary variable and is strongly imbalanced, with approximately 96% of observations in class 0 and 4% in class 1. As a binary feature, it does not contain outliers; instead, it reflects class imbalance.

---

# Multicollinearity Check
## Variance Inflation Factor (VIF)
## VIF Interpretation Guidelines

| VIF Value | Interpretation                              |
| --------- | ------------------------------------------- |
| 1         | No multicollinearity                        |
| 1 – 5     | Moderate correlation (generally acceptable) |
| > 5       | High multicollinearity                      |
| > 10      | Severe multicollinearity                    |

## Features Selected for VIF Analysis

Variance Inflation Factor (VIF) was computed using the numerical predictors in the dataset after cleaning and feature engineering. Only continuous and count-based numerical variables were included, as VIF is not appropriate for categorical features without encoding.

The following features were selected for analysis:

* age
* duration
* campaign
* pdays
* previous
* emp.var.rate
* cons.price.idx
* cons.conf.idx
* euribor3m
* nr.employed
* previously_contacted

Both `pdays` and `previously_contacted` were initially included for exploratory purposes; however, due to their deterministic relationship, they are expected to exhibit strong collinearity.

## Compute VIF Scores

In [51]:
# Copy final dataset after preprocessing
df_vif = df_pdays.copy()

# VIF numeric features only
vif_cols = [
    "age",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
    "previously_contacted"
]

# Select data
X_vif = df_vif[vif_cols].copy()

# Ensure strict numeric format
X_vif = X_vif.apply(pd.to_numeric, errors="coerce")

# Remove missing values (required for VIF computation)
X_vif = X_vif.dropna()

# Compute VIF
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns

vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

# Sort results
vif_data = vif_data.sort_values(by="VIF", ascending=False)

# Final output
vif_data.round(4)

,Feature,VIF
9,nr.employed,25074.8132
6,cons.price.idx,21315.0819
8,euribor3m,222.6852
7,cons.conf.idx,119.4840
5,emp.var.rate,28.1366
0,age,16.3948
10,previously_contacted,3.8593
3,pdays,3.3367
2,campaign,2.0626
1,duration,2.0493


## Multicollinearity Interpretation (VIF Results)

### Severe multicollinearity

These variables show **very high VIF**, meaning they are strongly linearly dependent:

* `nr.employed` → 25074.81
* `cons.price.idx` → 21315.08
* `euribor3m` → 222.69
* `cons.conf.idx` → 119.48
* `emp.var.rate` → 28.14

These are all **macroeconomic indicators**, and they move together over time.

In practice, this means:

* They are highly redundant
* Keeping all of them can destabilize linear models (logistic regression especially)

### Moderate / acceptable multicollinearity

* `age` → 16.39 (moderately high, but often tolerated depending on model)
* `previously_contacted` → 3.86
* `pdays` → 3.34
* `campaign` → 2.06
* `duration` → 2.05
* `previous` → 2.03

These variables are:

* Mostly independent
* Safe for modeling
* No strong multicollinearity concerns

## Key insight

### Economic variables cluster together

* euribor3m
* nr.employed
* emp.var.rate
* cons.price.idx
* cons.conf.idx

They form a **highly correlated macroeconomic block**.

## Modeling decision options

You have 3 valid options:

### Option A — Keep all (tree-based models only)

* Fine for Random Forest / XGBoost
* Multicollinearity not a problem

### Option B — Remove redundancy (recommended for linear models)

Keep only 1–2 from the economic group, for example:

* Keep: `euribor3m`
* Drop: `nr.employed`, `cons.price.idx`, `emp.var.rate`, `cons.conf.idx`

### Option C — Dimensionality reduction

* Apply PCA to economic variables
* Replace them with 1–2 components

## Conclusion

The VIF analysis reveals severe multicollinearity among macroeconomic indicators (`nr.employed`, `cons.price.idx`, `euribor3m`, `cons.conf.idx`, `emp.var.rate`). This is expected due to their shared economic origin and temporal dependency. Other features exhibit acceptable VIF values, indicating low to moderate multicollinearity.

To ensure model stability and compatibility across different benchmarking algorithms, a simplified feature selection strategy is applied by removing redundant macroeconomic variables while retaining only the most representative indicator (`euribor3m`). This approach reduces multicollinearity while preserving essential economic information in a compact and interpretable form.

# **Construction and Export of the Final Modeling Dataset**

After completing data cleaning, feature engineering, and multicollinearity analysis, the final modeling dataset is constructed. This step ensures that all preprocessing decisions are consistently applied, including the removal of highly collinear macroeconomic variables identified through VIF analysis and the transformation of key features such as `pdays`.

The resulting dataset is saved as a clean and reproducible version for use in model training and benchmarking across different algorithms.

In [62]:
# Saving a copy of the clean dataset
import os

os.makedirs("../data/processed/", exist_ok=True)

# Features removed based on VIF analysis (Option B)
cols_to_drop = [
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "nr.employed"
]

df_final = df_vif.drop(columns=cols_to_drop)

# Save final dataset
df_final.to_csv("../data/processed/bank_additional_clean.csv", index=False)

In [3]:
dataset_path = "../data/processed/bank_additional_clean.csv"
df_clean = pd.read_csv(dataset_path)

In [4]:
df_clean.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,euribor3m,y,previously_contacted
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,487,2,0,0,nonexistent,1.313,no,0
1,39,services,single,high.school,no,no,no,telephone,may,fri,346,4,0,0,nonexistent,4.855,no,0
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,227,1,0,0,nonexistent,4.962,no,0
3,38,services,married,basic.9y,no,yes,no,telephone,jun,fri,17,3,0,0,nonexistent,4.959,no,0
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,58,1,0,0,nonexistent,4.191,no,0


---

# Planned Feature Preprocessing (Updated After Cleaning & VIF Analysis)

| Column                 | Variable Type       | Final Treatment                                  | Encoding / Scaling Strategy       | Reason                                              |
| ---------------------- | ------------------- | ------------------------------------------------ | --------------------------------- | --------------------------------------------------- |
| `age`                  | Numerical           | Kept                                             | RobustScaler                      | Mild skewness and outliers                          |
| `job`                  | Nominal Categorical | Kept (unknown retained as category)              | One-Hot Encoding (`drop='first'`) | Low frequency unknown values, informative signal    |
| `marital`              | Nominal Categorical | Imputed missing values with mode                 | One-Hot Encoding (`drop='first'`) | Very low missing rate                               |
| `education`            | Ordinal Categorical | Kept (unknown retained)                          | OrdinalEncoder                    | Natural ordering preserved                          |
| `default`              | Nominal Categorical | Unknown retained as category                     | One-Hot Encoding (`drop='first'`) | High proportion of unknown values                   |
| `housing`              | Nominal Categorical | Imputed missing values with mode                 | One-Hot Encoding (`drop='first'`) | Low missing rate                                    |
| `loan`                 | Nominal Categorical | Imputed missing values with mode                 | One-Hot Encoding (`drop='first'`) | Low missing rate                                    |
| `contact`              | Nominal Categorical | Kept                                             | One-Hot Encoding (`drop='first'`) | Binary categorical                                  |
| `month`                | Ordinal Categorical | Kept                                             | OrdinalEncoder                    | Temporal structure                                  |
| `day_of_week`          | Ordinal Categorical | Kept                                             | OrdinalEncoder                    | Temporal structure                                  |
| `duration`             | Numerical           | Kept for benchmarking models                     | RobustScaler                      | High skew + potential leakage risk (benchmark only) |
| `campaign`             | Numerical           | Kept                                             | RobustScaler                      | Right-skewed distribution                           |
| `pdays`                | Numerical           | Transformed (`999 → 0`) + binary feature created | RobustScaler                      | Rare event encoding handled explicitly              |
| `previous`             | Numerical           | Kept                                             | RobustScaler                      | Low multicollinearity                               |
| `previously_contacted` | Binary Feature      | Derived from `pdays`                             | No scaling needed                 | Captures contact history explicitly                 |
| `poutcome`             | Nominal Categorical | Kept                                             | One-Hot Encoding (`drop='first'`) | No natural order                                    |
| `emp.var.rate`         | Numerical           | **Removed**                                      | —                                 | High multicollinearity                              |
| `cons.price.idx`       | Numerical           | **Removed**                                      | —                                 | High multicollinearity                              |
| `cons.conf.idx`        | Numerical           | **Removed**                                      | —                                 | High multicollinearity                              |
| `euribor3m`            | Numerical           | Kept                                             | MinMaxScaler                      | Representative macroeconomic indicator              |
| `nr.employed`          | Numerical           | **Removed**                                      | —                                 | Redundant macroeconomic signal                      |
| `y`                    | Target Variable     | Converted to binary                              | LabelEncoder                      | Required for classification                         |


## Key changes we successfully incorporated

* ✔ Removed redundant macroeconomic variables (VIF-based decision)
* ✔ Kept only `euribor3m` as representative economic signal
* ✔ Formalized `pdays → previously_contacted`
* ✔ Maintained dual modeling logic for `duration`
* ✔ Clean separation between encoding strategies

---

# Model Building

To prepare the dataset for machine learning, the features were divided into groups according to their data type and preprocessing requirements.

This structure allows the preprocessing pipeline to apply appropriate transformations to each variable type before training the models.

The preprocessing strategy includes:

- Scaling numerical variables using the selected scaler.
- Ordinal encoding for ordered categorical variables
- One-hot encoding for nominal categorical variables
- Encoding the target variable using `LabelEncoder`

The following feature groups were used to build the preprocessing pipeline and classification models:

In [5]:
# =========================
# Feature Groups
# =========================

# Numerical features

# Uniform numerical features (no strong skew, no outliers)
minmax_features = [
    "euribor3m"
]

# Normal numerical features (approximately symmetric, safe for StandardScaler if needed)
standard_features = [
]

# Skewed numerical features (outliers + heavy tails → RobustScaler)
robust_features = [
    "age",
    "duration",
    "campaign",
    "pdays",
    "previous"
]

# Categorical features

# Ordinal categorical features
ordinal_features = [
    "education",
    "month",
    "day_of_week"
]

# Ordinal categories (explicit ordering)

ordinal_categories = [
    [
        "illiterate",
        "basic.4y",
        "basic.6y",
        "basic.9y",
        "high.school",
        "professional.course",
        "university.degree"
    ],
    [
        "mar",
        "apr",
        "may",
        "jun",
        "jul",
        "aug",
        "sep",
        "oct",
        "nov",
        "dec"
    ],
    [
        "mon",
        "tue",
        "wed",
        "thu",
        "fri"
    ]
]

# Nominal categorical features
nominal_features = [
    "job",
    "marital",
    "default",
    "housing",
    "loan",
    "contact",
    "poutcome"
]

# Features to remove (based on VIF + feature engineering decisions)
drop_features = [
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "nr.employed"
]

# Target
target = "y"

# Binary classification mapping
target_map = {
    "no": 0,
    "yes": 1
}

## Numerical Pipeline

The numerical preprocessing pipeline handles missing values and feature scaling before model training.

This scaler transforms features into a common range while preserving the original distribution shape and relative distances between observations.

Median imputation is included as a robust strategy to handle potential missing values during future predictions or deployment scenarios, since the median is less sensitive to outliers than the mean.

In [6]:
# =========================
# Pipelines
# =========================

minmax_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

standard_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

robust_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

# =========================
# Numerical Transformer
# =========================

num_transformer = ColumnTransformer(
    transformers=[
        ("minmax", minmax_pipeline, minmax_features),
        ("standard", standard_pipeline, standard_features),
        ("robust", robust_pipeline, robust_features)
    ]
)

num_transformer

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('minmax', ...), ('standard', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

## Ordinal Pipeline

The ordinal preprocessing pipeline is responsible for handling ordered categorical variables.

Missing values are imputed using the most frequent category because these variables contain a limited number of discrete ordered labels, making the mode a simple and consistent replacement strategy.

`OrdinalEncoder` is used to preserve the natural ranking of the categories. This is important because the values represent increasing levels of quality or condition, and converting them into arbitrary one-hot vectors would remove that ordered relationship.

In [7]:
# =========================
# Ordinal Pipeline
# =========================

ordinal_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "ordinal_encoder",
        OrdinalEncoder(
            categories=ordinal_categories,
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])

ordinal_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('ordinal_encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite im

## Nominal Pipeline

The nominal preprocessing pipeline handles categorical variables without a natural order.

Missing values are replaced using the most frequent category to maintain consistency within each feature.

`OneHotEncoder` is applied to transform categories into binary indicator variables. The option `drop="first"` is used to remove one redundant category from each feature, helping reduce multicollinearity in the model. Additionally, `handle_unknown="ignore"` ensures that unseen categories during prediction do not cause errors in the pipeline.

In [8]:
# =========================
# Nominal Pipeline
# =========================
nominal_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "onehot",
        OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

nominal_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('onehot', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation.

## Column Transformer

The `ColumnTransformer` combines all preprocessing pipelines into a single unified preprocessing step.

Each group of variables is processed according to its data type and characteristics:

- Numerical variables are imputed and scaled
- Ordinal categorical variables are encoded while preserving category order
- Nominal categorical variables are transformed using one-hot encoding

This approach ensures that all preprocessing steps are applied consistently during both training and prediction, helping prevent data leakage and simplifying the machine learning workflow.

Using a centralized preprocessor also makes the pipeline easier to maintain, reproduce, and deploy.

In [ ]:
# =========================
# Column Transformer
# =========================

preprocessor = ColumnTransformer(
    transformers=[
        ("minmax_num", 
        minmax_pipeline, 
        minmax_features),

        ("standard_num",
        standard_pipeline,
        standard_features),

        ("robust_num",
        robust_pipeline,
        robust_features),

        ("ord",
        ordinal_pipeline,
        ordinal_features),

        ("nom",
        nominal_pipeline,
        nominal_features)
    ],
    sparse_threshold=0,
    remainder="drop"
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('minmax_num', ...), ('standard_num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fea

## Full Modeling Pipeline

The complete modeling pipeline combines preprocessing and model training into a single workflow.

This structure ensures that all preprocessing steps are automatically applied before fitting the model, including:

- missing value imputation
- scaling of numerical features
- ordinal encoding
- one-hot encoding

The processed data is then passed directly to the model.

Using a unified pipeline improves reproducibility, prevents data leakage, and simplifies both training and future predictions, since the same transformations are consistently applied to new data.

In [11]:
# =========================
# Full Modeling Pipeline
# =========================

dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        DecisionTreeClassifier(random_state=42)
    )
])

dt_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('minmax_num', ...), ('standard_num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diff

## Train-Test Split

Before training the model, the dataset is divided into features (`X`) and target (`y`).

The target variable is `y`, while the remaining columns are used as predictive features. Columns previously identified as redundant or non-informative for modeling are removed from the dataset.

The data is then split into training and testing subsets using an 70/30 ratio:

- **70%** of the data is used to train the model
- **30%** is reserved for evaluating model performance on unseen data

A fixed `random_state` is used to ensure reproducibility of the results. 

Additionally, **stratified sampling** is recommended to preserve the original class distribution of `y` in both the training and testing sets.

In [13]:
# =========================
# Prepare Modeling Dataset
# =========================

df_model = df_clean.copy()

# Target
target = "y"

# Features and target
X = df_model.drop(columns=[target])
y = df_model[target].map(target_map)

# =========================
# Train-Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [14]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")

print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}")

X_train shape: (2883, 17)
X_test shape : (1236, 17)
y_train shape: (2883,)
y_test shape : (1236,)


### Shape check

* Train: (2883, 17)
* Test: (1236, 17)

- Consistent feature space
- No leakage
- Correct 70/30 split behavior

In [15]:
print("Overall")
print(y.value_counts(normalize=True))

print("\nTrain")
print(y_train.value_counts(normalize=True))

print("\nTest")
print(y_test.value_counts(normalize=True))

Overall
y
0    0.890507
1    0.109493
Name: proportion, dtype: float64

Train
y
0    0.890392
1    0.109608
Name: proportion, dtype: float64

Test
y
0    0.890777
1    0.109223
Name: proportion, dtype: float64


### Stratification check

#### Overall distribution

* Class 1: 68.73%
* Class 0: 31.27%

#### Train

* Class 1: 68.76%
* Class 0: 31.24%

#### Test

* Class 1: 68.65%
* Class 0: 31.35%

This is **excellent stratification quality**:

* Train ≈ Test ≈ Full dataset
* No sampling bias
* No class distortion

# Pipeline Preparation and Baseline Model Training
This section defines the final preprocessing pipeline and integrates it into a baseline machine learning model. The preprocessing steps are saved to ensure consistent transformations across all benchmarking models. A Decision Tree classifier is using the prepared pipeline, establishing a reference performance baseline for subsequent model comparisons.

In [17]:
# Save preprocessing pipeline for reuse across models
joblib.dump(preprocessor, "../models/preprocessor.joblib")

['../models/preprocessor.joblib']

In [18]:
preprocessor = joblib.load("../models/preprocessor.joblib")

In [19]:
# Build baseline model pipeline (Decision Tree)
model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('minmax_num', ...), ('standard_num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diff

---

# Conclusions and Insights

This stage consolidates the main findings from the data preprocessing and feature engineering pipeline applied to the Bank Marketing dataset. The dataset was successfully cleaned, standardized, and transformed into a modeling-ready format while preserving all observations and ensuring consistency across feature types.

Key data quality issues were addressed, including hidden missing values encoded as `"unknown"` and the special treatment of `pdays = 999`, which was transformed into a meaningful binary indicator (`previously_contacted`). This improved interpretability and reduced distortion in numerical distributions.

Feature engineering decisions were guided by statistical behavior and business meaning. Numerical variables were scaled using appropriate strategies (MinMax, Standard, or Robust Scaling) depending on their distribution characteristics and presence of outliers. Categorical variables were encoded using a combination of One-Hot Encoding and Ordinal Encoding, ensuring that both nominal and ordered relationships were properly represented.

Multicollinearity analysis revealed severe redundancy among macroeconomic indicators. Based on Variance Inflation Factor (VIF) results, a feature selection decision was made to retain only `euribor3m`, while highly collinear variables such as `nr.employed`, `cons.price.idx`, `cons.conf.idx`, and `emp.var.rate` were removed. This reduction improved model stability and reduced redundancy without significantly compromising informational value.

The final dataset was standardized into a reusable preprocessing pipeline, ensuring reproducibility and consistency across all future models.

Overall, the prepared dataset is now fully compatible with multiple machine learning algorithms and provides a solid foundation for systematic model comparison and performance optimization.

---

# Appendix

## References

- [Bank Marketing Dataset from the UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing)
- Machine Learning notebooks from the bootcamp repository (*sonda2026* branch)
- Machine Learning notebooks from the shared Assistanship Google Drive folder
- [Pandas Documentation](https://pandas.pydata.org/docs/user_guide/index.html?utm_source=chatgpt.com)
- [Plotly Documentation](https://plotly.com/python/?utm_source=chatgpt.com)

## Acknowledgements

I would like to thank my instructors for their guidance, continuous support, and encouragement throughout the development of this project.

I also acknowledge the use of AI-assisted tools to support debugging, code review, documentation, and the exploration of machine learning concepts during the analysis process. All modeling decisions, interpretations, and conclusions presented in this work were made independently and remain my own responsibility.

---

### Tools and Technologies

- Python
- Pandas
- NumPy
- SciPy
- Plotly
- SkLearn
- JobLib
- Jupyter Notebook